In [1]:
import os
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/NeuronWireTracingEngine'

In [33]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelBuildEncConfig:
    root_dir: Path
    train_img: Path
    label_pth:Path
    enc_model: Path
    params_box_size: list
    params_batchSize: int
    params_outputSize: int
    params_epoch: int
    params_lr: float
    params_augmentT: bool
    params_augmentL:bool 

In [3]:
from src.neuronTracer import *
from src.neuronTracer.constants import *
from src.neuronTracer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_build_enc_config(self) -> ModelBuildEncConfig:
        config = self.config.model_build_enc

        create_directories([config.root_dir])

        model_build_enc_config = ModelBuildEncConfig(
            root_dir= Path(config.root_dir),
            train_img= Path(config.train_img),
            label_pth= Path(config.label_pth),
            enc_model= Path(config.enc_model),
            params_box_size= tuple(self.params.box_size_mac),
            params_batchSize  = int(self.params.BATCH_SIZE),
            params_outputSize = int(self.params.OUTPUT_SIZE),
            params_augmentT= bool(self.params.augmentTrain),
            params_augmentL= bool(self.params.augmentLabel),

        )

        return model_build_enc_config

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

import multiprocessing


In [ ]:
class modelDiagnostic:
    def __init__(self, config:ModelBuildEncConfig):
        self.config=config
        self.image_dir = config.train_img
        self.label_dir = config.label_pth
        self.img_files = sorted(self.image_dir.glob('box_*.pt'))
        self.lbl_files = sorted(self.image_dir.glob('box_*.pt'))
        
        

    def verifyPair(self):
        """Verify pairing
        You will be prompted twice — once for image files, once for label files.
        """
        img_files = self.img_files
        lbl_files = self.lbl_files

        logger.info(f'\nImage files : {[f.name for f in img_files]}')
        logger.info(f'\nLabel files : {[f.name for f in lbl_files]}')

        assert len(img_files) > 0,           'No image files found — re-run this cell'
        assert len(img_files) == len(lbl_files), 'Image / label count mismatch'
        for i, l in zip(img_files, lbl_files):
            assert i.name == l.name, logger.info(f'Filename mismatch: {i.name} vs {l.name}')

        logger.info(f'All {len(img_files)} pairs matched ✅')
    
    def sysConfig(self):
        logger.info("🔍 --- Project Diagnostic ---")

        # 1. Multiprocessing Check
        # Colab uses Linux, which defaults to 'fork'. 
        # For CUDA, 'spawn' is technically safer to avoid deadlocks.
        logger.info("[MULTIPROCESSING]")
        try:
            start_method = multiprocessing.get_start_method()
            logger.info(f"  Start method : {start_method}")
            cpu_count = multiprocessing.cpu_count()
            logger.info(f"  CPU cores    : {cpu_count} logical")
        except Exception as e:
            logger.error(f"  Multiprocessing check failed: {e}")

        # 2. Hardware & Backend Check
        logger.info("[DEVICE & BACKEND]")
        
        # NVIDIA CUDA (Colab Standard)
        if torch.cuda.is_available():
            device = torch.device("cuda")
            prop = torch.cuda.get_device_properties(0)
            
            logger.info(f"  Backend      ⚙️ : CUDA (NVIDIA)")
            logger.info(f"  GPU Model    : {prop.name}")
            logger.info(f"  VRAM Total   : {prop.total_memory / 1e9:.1f} GB")
            
            # Check for 'Compute Capability' (DINOv2 runs best on 7.0+)
            cc = f"{prop.major}.{prop.minor}"
            logger.info(f"  Compute Cap  : {cc}")
            
            # Check Memory Fragmentation
            vram_reserved = torch.cuda.memory_reserved(0) / 1e9
            vram_allocated = torch.cuda.memory_allocated(0) / 1e9
            logger.info(f"  VRAM Reserved: {vram_reserved:.1f} GB")
            logger.info(f"  VRAM Active  : {vram_allocated:.1f} GB")

            # Smoke Test
            try:
                test_tensor = torch.zeros((100, 100), device=device)
                logger.info("  Smoke test 💨 : CUDA Tensor creation OK")
                del test_tensor # Clean up immediately
            except Exception as e:
                logger.warning(f"  Smoke test 💨 : FAILED — {e}")

        # Apple Silicon (For when you run locally)
        elif torch.backends.mps.is_available():
            logger.info("  Backend      ⚙️ : Apple Silicon (MPS)")
            logger.info("  Status       ✅: OK")

        else:
            logger.warning("  Backend      ⚙️ : CPU only — No Accelerator Found")

        # 3. Environment Context (Colab Specific)
        if 'COLAB_GPU' in os.environ:
            logger.info("  Environment  🌐: Google Colab detected")
        
        return device

    def dataInspect(self):
        '''
        Confirms the shape, dtype, and value range before anything is built.
        '''
        sample_img = torch.load(self.img_files[0], weights_only=True)
        sample_lbl = torch.load(self.lbl_files[0], weights_only=True)

        logger.info('=== Image (box_0000.pt from images/) ===')
        logger.info(f'  shape  : {sample_img.shape}')    # expect [1, Z, H, W] or [Z, H, W]
        logger.info(f'  dtype  : {sample_img.dtype}')
        logger.info(f'  range  : [{sample_img.min():.3f}, {sample_img.max():.3f}]')

        logger.info('=== Label (box_0000.pt from labels/) ===')
        logger.info(f'  shape  : {sample_lbl.shape}')
        logger.info(f'  dtype  : {sample_lbl.dtype}')
        logger.info(f'\n unique : {sample_lbl.unique().tolist()}')

        # DINOv2 operates on 2D slices — we extract one Z-slice per sample
        # The spatial dims (H, W) get resized to 224×224 inside DinoAdapter
        SPATIAL_Z = sample_img.shape[-3] if sample_img.dim() == 4 else sample_img.shape[-3]
        logger.info(f'Z-depth per volume : {SPATIAL_Z}')
        logger.info(f'Total 2D slices    : {len(self.img_files) * SPATIAL_Z}')

        

In [ ]:
try:
    config = ConfigurationManager()
    model_enc_config = config.get_model_build_enc_config()
    diagnosis = modelDiagnostic(config=model_enc_config)
    diagnosis.verifyPair()
    device = diagnosis.sysConfig()
    diagnosis.dataInspect()
except Exception as e:
    raise e

[2026-03-23 22:44:26,369: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-23 22:44:26,372: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-23 22:44:26,373: INFO: common: created directory at: artifacts]
[2026-03-23 22:44:26,374: INFO: common: created directory at: artifacts/model_build]
[2026-03-23 22:44:26,443: INFO: 1999881544: 
Image files : ['box_0000.pt', 'box_0001.pt', 'box_0002.pt', 'box_0003.pt', 'box_0004.pt', 'box_0005.pt', 'box_0006.pt', 'box_0007.pt', 'box_0008.pt', 'box_0009.pt', 'box_0010.pt', 'box_0011.pt', 'box_0012.pt', 'box_0013.pt', 'box_0014.pt', 'box_0015.pt', 'box_0016.pt', 'box_0017.pt', 'box_0018.pt', 'box_0019.pt', 'box_0020.pt', 'box_0021.pt', 'box_0022.pt', 'box_0023.pt', 'box_0024.pt', 'box_0025.pt', 'box_0026.pt', 'box_0027.pt', 'box_0028.pt', 'box_0029.pt', 'box_0030.pt', 'box_0031.pt', 'box_0032.pt', 'box_0033.pt', 'box_0034.pt', 'box_0035.pt', 'box_0036.pt', 'box_0037.pt', 'box_0038.pt', 'box_0039.pt', 'box_00

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

/Users/kiranprasadjp/Documents/Pros/NeuronWireTracingEngine/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
class Adapter(nn.Module):
    """
    1. The Adapter (same name as original)
        Converts [B, 1, H, W] grayscale → [B, 3, 224, 224] fake-RGB for DINOv2
    """
    def __init__(self):
        super().__init__()
        self.resize = nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.resize(x)         # [B, 1, 224, 224]
        x = x.repeat(1, 3, 1, 1)  # [B, 3, 224, 224]  — fake RGB
        return x

class BTDecoder(nn.Module):
    '''
    # ─────────────────────────────────────────────────────────────────────────────
    # 2. The Botanist Decoder 
    #
    # FIX: original took flat CLS token [B, 768] → view(B,768,1,1) — 1x1 spatial!
    # Now takes 256 patch tokens [B, 256, 768] → reshape [B, 768, 16, 16] — real map.
    #
    # FIX: Sigmoid removed. Output is raw logits.
    #      Use BCEWithLogitsLoss (stable). Apply sigmoid only at inference.
    # ─────────────────────────────────────────────────────────────────────────────
    '''
    PATCH_GRID = 16   # 224 // 14 = 16 patches per side
    def __init__(self):
        super().__init__()
        self.dino_dim = 768

        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(768, 256, kernel_size=4, stride=4),   # 16 → 64
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2),    # 64 → 128
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.deconv3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),     # 128 → 256
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final = nn.Sequential(
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
            # NO Sigmoid — BCEWithLogitsLoss fuses it stably
        )

    def forward(self, patch_tokens):
        # patch_tokens: [B, 256, 768]
        B = patch_tokens.shape[0]
        x = patch_tokens.transpose(1, 2).reshape(B, self.dino_dim,
                                                  self.PATCH_GRID, self.PATCH_GRID)
        x = self.deconv1(x)   # [B, 256,  64,  64]
        x = self.deconv2(x)   # [B,  64, 128, 128]
        x = self.deconv3(x)   # [B,  32, 256, 256]
        return self.final(x)  # [B,   1, 224, 224]  raw logits

class DinoEncoder(nn.Module):
    """
    # ─────────────────────────────────────────────────────────────────────────────
    # 3. The Master Model (same name as original)
    # Frozen DINOv2 brain + unfrozen BotanistDecoder
    # ─────────────────────────────────────────────────────────────────────────────
    """
    def __init__(self):
        super().__init__()
        self.adapter = Adapter()

        print('Downloading DINOv2-ViT-B/14 from Meta (~330 MB, once) ...')
        self.dino_encoder = torch.hub.load(
            'facebookresearch/dinov2',
            'dinov2_vitb14',
            trust_repo=True,
        )

        # Freeze the entire DINOv2 brain
        for param in self.dino_encoder.parameters():
            param.requires_grad = False
        self.dino_encoder.eval()
        print(f'  DINOv2 frozen ({sum(p.numel() for p in self.dino_encoder.parameters())/1e6:.0f}M params)')

        self.botanist_decoder = BTDecoder()
        print(f'  Decoder trainable ({sum(p.numel() for p in self.botanist_decoder.parameters())/1e6:.1f}M params)')

    def forward(self, x):
        # x: [B, 1, H, W]  grayscale EM slice
        x = self.adapter(x)                                    # [B, 3, 224, 224]

        # FIX: use patch tokens (256 spatial tokens), NOT the CLS token
        with torch.no_grad():
            features = self.dino_encoder.get_intermediate_layers(x, n=1)
        patch_tokens = features[0]                             # [B, 256, 768]

        return self.botanist_decoder(patch_tokens)             # [B, 1, 224, 224]


In [ ]:
try:
    config = ConfigurationManager()
    model_enc_config = config.get_model_build_enc_config()
    encoder = DinoEncoder()
    # logger.info(encoder.state_dict())

except Exception as e:
    raise e

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── 1. The Dataset Class ──────────────────────────────────────────────────────
class BoxPtDataset(Dataset):
    """
    Reads box_NNNN.pt pairs. Expects direct lists of files so we can easily 
    split them into Train and Validation sets later.
    """
    def __init__(self, img_files, lbl_files, out_size=224, augment=False):
        self.samples = []
        self.img_files = img_files
        self.lbl_files = lbl_files
        self.out_size = out_size
        self.augment = augment

        # Index every Z-slice from every volume
        for img_path, lbl_path in zip(self.img_files, self.lbl_files):
            # We use weights_only=True for security, as recommended by PyTorch
            vol = torch.load(img_path, map_location='cpu', weights_only=True)
            if vol.dim() == 3:   
                vol = vol.unsqueeze(0)
            
            n_z = vol.shape[1]
            for z in range(n_z):
                self.samples.append((img_path, lbl_path, z))

        logger.info(f'BoxPtDataset initialized: {len(self.samples)} total slices from {len(self.img_files)} volumes.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path, z = self.samples[idx]

        img = torch.load(img_path, map_location='cpu', weights_only=True).float()
        lbl = torch.load(lbl_path, map_location='cpu', weights_only=True).float()

        # Ensure [1, Z, H, W]
        if img.dim() == 3: img = img.unsqueeze(0)
        if lbl.dim() == 3: lbl = lbl.unsqueeze(0)

        img_slice = img[:, z, :, :]   # [1, H, W]
        lbl_slice = lbl[:, z, :, :]   # [1, H, W]

        # 1. Normalize image to [0, 1]
        mn, mx = img_slice.min(), img_slice.max()
        img_slice = (img_slice - mn) / (mx - mn + 1e-6)

        # 2. THE PHOTOCOPY FIX: Convert 401 unique instance IDs into binary cell walls {0, 1}
        # Assuming 1 is the cell boundary wall we want the AI to draw
        lbl_slice = (lbl_slice > 0).float()

        # 3. Resize label to 224x224 to match the DINOv2 / Decoder output
        lbl_slice = F.interpolate(
            lbl_slice.unsqueeze(0),
            size=(self.out_size, self.out_size),
            mode='nearest',
        ).squeeze(0)   # [1, 224, 224]

        # 4. Augmentation (training only)
        if self.augment:
            if torch.rand(1) > 0.5:
                img_slice = torch.flip(img_slice, [1])   # flip H
                lbl_slice = torch.flip(lbl_slice, [1])
            if torch.rand(1) > 0.5:
                img_slice = torch.flip(img_slice, [2])   # flip W
                lbl_slice = torch.flip(lbl_slice, [2])
            
            # Intensity jitter
            scale = 0.9 + 0.2 * torch.rand(1).item()
            img_slice = (img_slice * scale).clamp(0.0, 1.0)

        return img_slice, lbl_slice

# ── 2. The Dataloader Factory ─────────────────────────────────────────────────
def build_dataloaders(config):
    """
    Takes your ConfigurationManager config, splits the files, 
    and returns ready-to-use PyTorch DataLoaders.
    """
    logger.info('Locating files and building datasets...')
    
    # Grab the paths from the config
    image_dir = config.train_img
    label_dir = config.label_pth  
    
    # Safely get all files (Fixed the label_dir typo here)
    img_files = sorted(image_dir.glob('box_*.pt'))
    lbl_files = sorted(label_dir.glob('box_*.pt'))
    
    assert len(img_files) == len(lbl_files), "Mismatch between number of images and labels!"
    assert len(img_files) > 1, "Need at least 2 files to create a train/val split!"

    # Split: all but last volume = train, last volume = val
    train_img_files = img_files[:-1]
    train_lbl_files = lbl_files[:-1]
    val_img_files   = img_files[-1:]
    val_lbl_files   = lbl_files[-1:]

    # Instantiate the Datasets using the fixed arguments
    train_ds = BoxPtDataset(
        img_files=train_img_files, 
        lbl_files=train_lbl_files, 
        out_size=config.params_outputSize,
        augment=config.params_augmentT
    )
    
    val_ds = BoxPtDataset(
        img_files=val_img_files,   
        lbl_files=val_lbl_files,  
        out_size=config.params_outputSize,
        augment=config.params_augmentL # Assuming this is meant to be False for validation
    )

    # Wrap them in DataLoaders
    train_loader = DataLoader(
        train_ds, 
        batch_size=config.params_batchSize, 
        shuffle=True,
        num_workers=0, 
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_ds,   
        batch_size=config.params_batchSize, 
        shuffle=False,
        num_workers=0
    )

    # Verification printout
    images, labels = next(iter(train_loader))
    logger.info(f'\n[Data Flow Verification]')
    logger.info(f'Image Batch Shape: {list(images.shape)} (expect: [{config.params_batchSize}, 1, 64, 64])')
    logger.info(f'Label Batch Shape: {list(labels.shape)} (expect: [{config.params_batchSize}, 1, 224, 224])')
    logger.info(f'Data Range: Min={images.min():.2f}, Max={images.max():.2f}')
    logger.info(f'Label unique values (Should only be 0.0 and 1.0): {labels.unique().tolist()}\n')

    return train_loader, val_loader

In [51]:
try:
    config = ConfigurationManager()
    model_enc_config = config.get_model_build_enc_config()
    train_loader, val_loader = build_dataloaders(model_enc_config)


except Exception as e:
    raise e

[2026-03-24 00:28:16,008: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-24 00:28:16,012: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 00:28:16,012: INFO: common: created directory at: artifacts]
[2026-03-24 00:28:16,014: INFO: common: created directory at: artifacts/model_build]
[2026-03-24 00:28:16,014: INFO: 1948108006: Locating files and building datasets...]
[2026-03-24 00:28:18,503: INFO: 1948108006: BoxPtDataset initialized: 25081 total slices from 3583 volumes.]
[2026-03-24 00:28:18,504: INFO: 1948108006: BoxPtDataset initialized: 7 total slices from 1 volumes.]
[2026-03-24 00:28:18,511: INFO: 1948108006: 
[Data Flow Verification]]
[2026-03-24 00:28:18,511: INFO: 1948108006: Image Batch Shape: [4, 1, 64, 64] (expect: [4, 1, 64, 64])]
[2026-03-24 00:28:18,512: INFO: 1948108006: Label Batch Shape: [4, 1, 224, 224] (expect: [4, 1, 224, 224])]
[2026-03-24 00:28:18,512: INFO: 1948108006: Data Range: Min=0.00, Max=1.00]
[2026-03-24 